# Hardware Analysis and YOLO26 Density Test Setup

This notebook analyzes your system hardware and creates an optimized configuration for the test-setup-03-run-density experiment.

## Your Optimal Hardware Configuration:
- **GPU Memory**: 23.99 GB (RTX 4090 equivalent)
- **Batch Size**: 32 (for efficient GPU utilization)
- **Image Size**: 1280x1280 (full resolution)
- **Video Stride**: 1 (process every frame)
- **Half Precision (FP16)**: Disabled (not needed with 24GB VRAM)
- **TF32 Mode**: Disabled (use standard precision)
- **Cache Clear Interval**: Every 10 batches

## Expected Performance:
- **Processing Speed**: Baseline 1.0x (no stride reduction needed)
- **Memory Efficiency**: 0% overhead (using full precision)
- **Suitable For**: Density analysis, aggregate statistics, crowd counting, high-quality inference


## Fix: OpenMP Conflict

Run this cell first if you get 'libiomp5md.dll already initialized' errors.

In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
print("OpenMP conflict fix: ENABLED")

OpenMP conflict fix: ENABLED


## Step 1: Analyze System Hardware

In [4]:
import json
import os
import platform
import shutil
from pathlib import Path

try:
    import psutil
except ImportError:
    os.system('pip install psutil -q')
    import psutil

import torch

# Get system information
cpu_count = os.cpu_count() or 1
psutil_cpu_count = psutil.cpu_count(logical=False) or cpu_count
ram = psutil.virtual_memory()

system_info = {
    "OS": platform.system(),
    "Platform": platform.platform(),
    "Processor": platform.processor(),
    "Physical CPUs": psutil_cpu_count,
    "Logical CPUs (threads)": cpu_count,
    "RAM Total (GB)": round(ram.total / (1024**3), 2),
    "RAM Available (GB)": round(ram.available / (1024**3), 2),
    "RAM Used (GB)": round(ram.used / (1024**3), 2),
    "RAM Percent": f"{ram.percent}%",
}

print("=" * 70)
print("SYSTEM HARDWARE ANALYSIS")
print("=" * 70)
for key, value in system_info.items():
    print(f"{key:.<30} {value}")

# GPU Information
print("\n" + "=" * 70)
print("GPU INFORMATION")
print("=" * 70)
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        free_bytes, total_bytes = torch.cuda.mem_get_info(i)
        print(f"  VRAM Total: {total_bytes / 1024**3:.2f} GB")
        print(f"  VRAM Free:  {free_bytes / 1024**3:.2f} GB")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Max Threads per Block: {getattr(props, 'max_threads_per_block', 'N/A')}")
else:
    print("No CUDA-capable GPU detected. CPU inference will be used.")

print("\n" + "=" * 70)

SYSTEM HARDWARE ANALYSIS
OS............................ Windows
Platform...................... Windows-11-10.0.26200-SP0
Processor..................... AMD64 Family 25 Model 8 Stepping 2, AuthenticAMD
Physical CPUs................. 24
Logical CPUs (threads)........ 48
RAM Total (GB)................ 127.84
RAM Available (GB)............ 95.84
RAM Used (GB)................. 32.0
RAM Percent................... 25.0%

GPU INFORMATION
CUDA Available: True
CUDA Version: 12.4
Number of GPUs: 1

GPU 0: NVIDIA GeForce RTX 4090
  VRAM Total: 23.99 GB
  VRAM Free:  22.46 GB
  Compute Capability: 8.9
  Max Threads per Block: N/A



## Step 2: Recommend Optimal Parameters

In [5]:
# Estimate optimal batch size based on VRAM
gpu_memory_gb = 8  # Default assumption
if torch.cuda.is_available():
    _, total_bytes = torch.cuda.mem_get_info(0)
    gpu_memory_gb = total_bytes / 1024**3

# YOLO26-X Pose model VRAM requirements (approximate)
# Base model: ~4GB, per-frame in batch: ~200-300MB each at imgsz=1280
# At imgsz=1024: ~180-250MB per frame
# At imgsz=960: ~100-150MB per frame
# At imgsz=640: ~50-75MB per frame

if gpu_memory_gb >= 24:
    # Ultra high-end GPU: RTX 4090, A100, H100
    recommended_batch = 32
    recommended_imgsz = 1280
    recommended_vid_stride = 1
    allow_tf32 = False
    use_fp16 = False
elif gpu_memory_gb >= 16:
    # High-end GPU: RTX 3090, RTX 4080
    recommended_batch = 32
    recommended_imgsz = 1280
    recommended_vid_stride = 1
    allow_tf32 = False
    use_fp16 = False
elif gpu_memory_gb >= 12:
    # Mid-high-end GPU: RTX 3080, RTX 4070
    recommended_batch = 24
    recommended_imgsz = 1024
    recommended_vid_stride = 1
    allow_tf32 = True
    use_fp16 = False
elif gpu_memory_gb >= 8:
    # Mid-range GPU: RTX 3070, RTX 4060
    recommended_batch = 24
    recommended_imgsz = 960
    recommended_vid_stride = 2
    allow_tf32 = True
    use_fp16 = True
elif gpu_memory_gb >= 6:
    # Budget GPU: RTX 3060
    recommended_batch = 12
    recommended_imgsz = 768
    recommended_vid_stride = 2
    allow_tf32 = True
    use_fp16 = True
else:
    # Low-end GPU or CPU inference
    recommended_batch = 8
    recommended_imgsz = 640
    recommended_vid_stride = 3
    allow_tf32 = True
    use_fp16 = True

recommendations = {
    "GPU Memory (GB)": round(gpu_memory_gb, 2),
    "Recommended Batch Size": recommended_batch,
    "Recommended Image Size": recommended_imgsz,
    "Recommended Video Stride": recommended_vid_stride,
    "Enable FP16 (half precision)": use_fp16,
    "Enable TF32 Mode": allow_tf32,
    "Cache Clear Interval (batches)": max(10, 100 // recommended_batch),
}

print("\n" + "=" * 70)
print("RECOMMENDED HARDWARE CONFIGURATION")
print("=" * 70)
for key, value in recommendations.items():
    print(f"{key:.<40} {value}")

print("\n" + "=" * 70)
print("BENEFITS OF THIS CONFIGURATION:")
print("=" * 70)

# Calculate speedup more accurately
if recommended_vid_stride == 1:
    speedup_factor = 1.0
else:
    speedup_factor = recommended_vid_stride * (0.9 if gpu_memory_gb >= 8 else 0.7)

memory_savings = 50 if use_fp16 else 0

print(f"✓ Processing Speed Improvement: {speedup_factor:.1f}x faster")
print(f"✓ Memory Efficiency: {memory_savings}% VRAM savings from FP16")
print(f"✓ Batch Processing: {recommended_batch} frames per inference (efficient GPU utilization)")
print(f"✓ Suitable for: Density analysis, aggregate statistics, crowd counting")


RECOMMENDED HARDWARE CONFIGURATION
GPU Memory (GB)......................... 23.99
Recommended Batch Size.................. 32
Recommended Image Size.................. 1280
Recommended Video Stride................ 1
Enable FP16 (half precision)............ False
Enable TF32 Mode........................ False
Cache Clear Interval (batches).......... 10

BENEFITS OF THIS CONFIGURATION:
✓ Processing Speed Improvement: 1.0x faster
✓ Memory Efficiency: 0% VRAM savings from FP16
✓ Batch Processing: 32 frames per inference (efficient GPU utilization)
✓ Suitable for: Density analysis, aggregate statistics, crowd counting


## Step 3: Create test-setup-03-run-density Directory and Copy Notebook

In [ ]:
# Define paths
notebooks_base = Path(r"C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks")
source_dir = notebooks_base / "test-setup-03-run-few"
target_dir = notebooks_base / "test-setup-03-run-density"
source_notebook = source_dir / "test-setup-03-run-few-yolo26-pose-count.ipynb"
target_notebook = target_dir / "test-setup-03-run-density-yolo26-pose-count.ipynb"

print(f"Creating directory: {target_dir}")
target_dir.mkdir(parents=True, exist_ok=True)
print("✓ Directory created")

if source_notebook.exists():
    print(f"\nCopying notebook from {source_notebook.name}...")
    shutil.copy(source_notebook, target_notebook)
    print(f"✓ Notebook copied to {target_notebook}")
else:
    print(f"✗ Source notebook not found: {source_notebook}")

## Step 4: Create Optimized data.yaml

In [ ]:
# Create optimized data.yaml
data_yaml_content = f"""# Optimized configuration for YOLO26 Pose person counting - Density Run
# Auto-generated based on detected hardware: {gpu_memory_gb:.1f}GB VRAM
# Configuration optimized for balanced performance and memory usage
# Paths are relative to this folder unless they are absolute.

app:
  name: "test-setup-03-run-density-yolo26-pose-count"
  install_requirements: false
  requirements_file: "../../requirements.txt"
  seed: 42

environment:
  YOLO_CONFIG_DIR: ".yolo"
  MPLCONFIGDIR: ".cache/matplotlib"
  TORCH_HOME: ".cache/torch"

paths:
  video: "input/20260329_34.mp4"
  weights: "yolo26x-pose.pt"
  weights_search_dirs:
    - "../"
    - "."
  output_dir: "output"
  annotated_video: "output/20260329_34_yolo26_pose_density_count.mp4"
  frame_counts_csv: "output/frame_counts_density.csv"
  summary_json: "output/summary_density.json"
  snapshots_dir: "output/snapshots_density"

runtime:
  require_cuda: true
  device: 0
  batch_size: {recommended_batch}
  auto_reduce_batch_on_oom: true
  empty_cuda_cache_every_batches: {max(10, 100 // recommended_batch)}
  torch_float32_matmul_precision: "high"
  allow_tf32: {str(allow_tf32).lower()}

inference:
  task: "pose"
  imgsz: {recommended_imgsz}
  conf: 0.20
  iou: 0.70
  max_det: 1000
  classes:
    - 0
  augment: false
  half: {str(use_fp16).lower()}
  verbose: false
  vid_stride: {recommended_vid_stride}

counting:
  count_source: "boxes"
  require_keypoints: false
  min_visible_keypoints: 3
  min_keypoint_conf: 0.25

output:
  save_annotated_video: true
  save_frame_counts: true
  save_summary: true
  save_snapshot_every_n_frames: 1440
  video_codec: "mp4v"
  progress_every_n_frames: 1440
  overlay:
    enabled: true
    font_scale: 0.8
    thickness: 2
"""

data_yaml_path = target_dir / "data.yaml"
data_yaml_path.write_text(data_yaml_content, encoding="utf-8")
print(f"✓ Created optimized data.yaml")
print(f"  Path: {data_yaml_path}")
print(f"\nConfiguration parameters:")
print(f"  Batch Size:     {recommended_batch}")
print(f"  Image Size:     {recommended_imgsz}")
print(f"  Video Stride:   {recommended_vid_stride}")
print(f"  Half Precision: {use_fp16}")
print(f"  TF32 Mode:      {allow_tf32}")

## Step 5: Copy Supporting Directories and Launch Notebook

In [ ]:
# Copy supporting directories from source if they exist
print("Copying supporting directories...")
for dir_name in [".yolo", ".cache", "input"]:
    source_item = source_dir / dir_name
    target_item = target_dir / dir_name
    
    if source_item.exists():
        if not target_item.exists():
            if source_item.is_dir():
                shutil.copytree(source_item, target_item)
                print(f"  ✓ Copied {dir_name}/")
            else:
                shutil.copy(source_item, target_item)
                print(f"  ✓ Copied {dir_name}")
        else:
            print(f"  ~ {dir_name}/ already exists")
    else:
        print(f"  - {dir_name}/ not found in source")

print(f"\n" + "=" * 70)
print("✓ SETUP COMPLETE!")
print("=" * 70)
print(f"Directory: {target_dir}")
print(f"Notebook:  {target_notebook}")
print(f"Config:    {data_yaml_path}")
print(f"\nTo run the notebook:")
print(f"  jupyter notebook \"{target_notebook}\"")